# Chapter 4: Transformer Architecture Types

The original 2017 Transformer had two halves: An **Encoder** and a **Decoder**. 
After the paper came out, scientists realized they didn't need both halves for every task. They ripped the architecture apart into three distinct families.

---

## 1. Encoder-Only (The Readers)

**Models:** BERT, RoBERTa, DistilBERT, SentenceTransformers.

**How it works:** 
It uses **Bidirectional Attention**. Every word can look at every single other word in the entire sentence (left and right). 

**Training Goal:** Masked Language Modeling (MLM). 
- Input: "The cat sat on the [MASK]."
- Output: The model must predict what the [MASK] is.

**What is it good for?**
Understanding deep, complex context. It "reads" the sentence globally.
- Sentiment Analysis (Is this tweet happy or sad?)
- Text Classification (Spam or Not Spam?)
- Semantic Search / Retrieval (Which document matches this query?)
*(It is terrible at writing long essays. It doesn't generate text well, it only predicts single masked words).*

---

## 2. Decoder-Only (The Writers)

**Models:** Every modern LLM. GPT-1/2/3/4, LLaMA, Claude, Mistral.

**How it works:** 
It uses **Causal (Masked) Attention**. A word can ONLY look at words that came *before* it. It is strictly forbidden from looking right into the future.

**Training Goal:** Causal Language Modeling (CLM) / Next-Token Prediction.
- Input: "The cat sat on the"
- Output: Predicts "mat". Then feeds "The cat sat on the mat" back into itself to predict the next word.

**What is it good for?**
Autoregressive Generation. Writing.
- Chatbots that write essays, code, poetry.
- Zero-shot instruction following.
*(They are historically slightly worse at reading comprehension than Encoders, but they scale so massively that 100B parameter Decoders essentially match 300M parameter Encoders at 'understanding' anyway).* 

---

## 3. Encoder-Decoder (The Translators)

**Models:** T5, BART, Original Transformer.

**How it works:** 
You keep both halves! 
- The Encoder uses Bidirectional Attention to perfectly "read" and understand the input text. It creates a massive matrix representing the meaning.
- The Decoder uses Causal Attention to "write" the output text letter-by-letter, while simultaneously using *Cross-Attention* to constantly peek at the Encoder's meaning matrix.

**What is it good for?**
Sequence-to-Sequence (Seq2Seq) tasks. Whenever the input is dramatically different from the output.
- Language Translation (English -> French).
- Text Summarization (500-page book -> 2-page summary).
*(T5 proved that you can literally frame ANY NLP task—even sentiment analysis—as a text-to-text sequence generation problem).* 

---

## 4. The Wildcard: Vision Transformers (ViT)

Wait, can we use Transformers for Images instead of CNNs (Convolutional Neural Networks)?

**Yes! (ViT - 2020)**
A Transformer expects a 1D sequence of "Word Tokens" (e.g., [142, 59, 102]). 
Images are 2D grids of pixels.

**How ViT works:**
1. Take a $224 \times 224$ image.
2. Cut it up into $16 \times 16$ pixel patches. (You now have 196 patches).
3. Flatten each patch into a 1D vector (using a simple linear projection).
4. Add positional encodings (so the model knows Patch 5 is the top-right corner).
5. Feed these 196 "patch tokens" into a standard Encoder-only Transformer (like BERT) exactly as if they were words in a sentence!

*(ViTs completely revolutionized Computer Vision, often beating out older ResNet CNN architectures when trained on massive datasets).* 

---

## 🎓 Interview Cheat Sheet

| Architecture | Model Examples | Attention Type | Primary Use Case |
| :--- | :---: | :---: | :--- |
| **Encoder-Only** | BERT, SBERT | Bidirectional | Semantic Search, Classification, Understanding |
| **Decoder-Only** | GPT-4, LLaMA | Causal (Masked) | Text Generation, Chatbots, Reasoning |
| **Encoder-Decoder** | T5, BART | Bidi + Causal + Cross | Summarization, Translation (Seq2Seq) |

---

### Q&A

**Q: Why don't we use BERT for text generation?**  
A: Because BERT was trained on Bidirectional Attention (looking left and right simultaneously). If you try to generate a word, it physically lacks the training to predict the "next" word relying purely on the left context. If you feed it "The cat sat", it cannot autoregress. It is designed to fill in blanks *between* words you already know.

**Q: What prevents a Decoder-only model from 'cheating' and looking at the next word during training?**  
A: The Causal Mask. It is a lower-triangular matrix of negative infinities added to the Attention Score matrix right before the Softmax operation. When $- \infty$ goes through Softmax, it becomes exactly $0 \%$. This physically blocks the math from allowing word $T$ to pull any Value data from word $T+1$.

**Q: In an Encoder-Decoder model like T5, what is Cross-Attention doing?**  
A: In the Decoder block, the self-attention layer first processes the currently generated output words. Then, the Cross-Attention layer kicks in: its *Query* matrix comes from the Decoder (what I am writing right now), while the *Keys* and *Values* matrices are permanently wired directly from the output of the final Encoder layer (what the source text meant). This forces the generation to stay faithful to the source material.